In [2]:
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

import pandas as pd
import numpy as np
import torch
import faiss
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer

In [3]:
!curl http://localhost:9200/

{
  "name" : "LAPTOP-UDLON1P4",
  "cluster_name" : "elasticsearch",
  "cluster_uuid" : "6eRXGDVkRza-7IQXW6AsSg",
  "version" : {
    "number" : "9.2.3",
    "build_flavor" : "default",
    "build_type" : "zip",
    "build_hash" : "d8972a71dbbd64ff17f2f4dba9ca2c3fe09fb100",
    "build_date" : "2025-12-16T10:09:08.849001802Z",
    "build_snapshot" : false,
    "lucene_version" : "10.3.2",
    "minimum_wire_compatibility_version" : "8.19.0",
    "minimum_index_compatibility_version" : "8.0.0"
  },
  "tagline" : "You Know, for Search"
}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   539  100   539    0     0   3731      0 --:--:-- --:--:-- --:--:--  3743


In [4]:
# instantiate Elasticsearch client localhost
client = Elasticsearch("http://localhost:9200")

In [5]:
# model from phase 2
model = SentenceTransformer("all-MiniLM-L6-v2") 

In [6]:
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [7]:
# index from phase 1
exists = client.indices.exists(index="my_idx")
print("Index exists: ", exists)

mapp= client.indices.get_mapping(index="my_idx")
print(mapp["my_idx"]["mappings"])

Index exists:  True
{'properties': {'doc_id': {'type': 'integer'}, 'text': {'type': 'text', 'analyzer': 'default', 'similarity': 'my_bm25'}}}


In [8]:
## reading the queries file
queries = pd.read_csv("IR2025/queries.csv")
print(queries.head())

    ID                                               Text
0  Q01  EUTRAVEL Optimodal European Travel Ecosystem E...
1  Q02  Track And Know Big Data for Mobility Tracking ...
2  Q03  SELIS, Towards a Shared European Logistics Int...
3  Q04  TYPHON Polyglot and Hybrid Persistence Archite...
4  Q05  CHARIOT Cognitive Heterogeneous Architecture f...


In [22]:
# parameters 
N = 200
a = 0.3 # for score

In [23]:
# defining the hybrid search function

def hybrid_search(query_text, k): 
    # using bm25 from elasticsearch from phase 1
    bm25_results = client.search(
        index="my_idx",
        size=N,
        query={
            "match": { "text": query_text }
        }
    )

    # saving retrieved documents (texts & ids) and their scores in lists
    docs = []
    doc_ids = []
    bm25_scores = []

    for hit in bm25_results["hits"]["hits"]:
        doc_ids.append(hit["_source"]["doc_id"])
        docs.append(hit["_source"]["text"])
        bm25_scores.append(hit["_score"])

    # normalizing bm25 scores to be in range [0,1]
    bm25_scores = np.array(bm25_scores)
    bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)

    # making embeddings for the N docs and the query from tranformers model from phase 2
    doc_embeddings = model.encode(
        docs,
        batch_size = 32,
        show_progress_bar=True,
        normalize_embeddings = True
    ).astype("float32")

    query_embedding = model.encode(
        [query_text],
        normalize_embeddings = True
    ).astype("float32")

    # making faiss index with doc embeddings
    dim = doc_embeddings.shape[1]  # embeddings dimention 384
    idx = faiss.IndexFlatIP(dim)    # making faiss index with inner product (cosine similarity)
    # we dont save ids now. we need to find the combined score first 
    idx.add(doc_embeddings)
    
    # executing the query search
    cosine_scores, faiss_indices = idx.search(query_embedding, N)
    cosine_scores = cosine_scores[0]
    faiss_indices = faiss_indices[0]  # faiss results = indices to ids
    
    # from faiss results
    selected_doc_ids = np.array(doc_ids)[faiss_indices]
    selected_bm25 = bm25_scores[faiss_indices]
        
    # normalizing cosine scores to be in range [0,1]
    cosine_scores = (cosine_scores - cosine_scores.min()) / (cosine_scores.max() - cosine_scores.min() + 1e-9)

    # σταθμισμένο μέσο όρο BM25 score και cosine similarity
    final_scores = a*cosine_scores + (1-a)*selected_bm25 

    # sorting the documents based on the combined score and return top k docs and scores
    top_k_idx = np.argsort(final_scores)[::-1][:k]

    final_doc_ids = np.array(selected_doc_ids)[top_k_idx] 
    final_scores = final_scores[top_k_idx]
    
    return final_doc_ids, final_scores
    

In [24]:
# for every query run search

def run_hybrid(queries, k):
    results = []

    for _, row in queries.iterrows():
        query_id = row["ID"]
        query_text = row["Text"]

        doc_ids, scores = hybrid_search(query_text, k)

        for rank, (doc_id, score) in enumerate(zip(doc_ids, scores), start=1):
            results.append({
                "query_id": query_id,
                "doc_id": int(doc_id),
                "rank": rank,
                "score": float(score)
            })

    return results
        

In [25]:
# running for k = 20 , 30 , 50 documents
results_20 = run_hybrid(queries, k=20)
print(results_20)
results_30 = run_hybrid(queries, k=30)
results_50 = run_hybrid(queries, k=50)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 7/7 [00:16<00:00,  2.32s/it]


[{'query_id': 'Q01', 'doc_id': 193378, 'rank': 1, 'score': 1.0000000119198482}, {'query_id': 'Q01', 'doc_id': 193373, 'rank': 2, 'score': 0.35789587453608174}, {'query_id': 'Q01', 'doc_id': 193715, 'rank': 3, 'score': 0.3210656597166335}, {'query_id': 'Q01', 'doc_id': 205685, 'rank': 4, 'score': 0.306915808957431}, {'query_id': 'Q01', 'doc_id': 206230, 'rank': 5, 'score': 0.3045921282922276}, {'query_id': 'Q01', 'doc_id': 193375, 'rank': 6, 'score': 0.29722598494913616}, {'query_id': 'Q01', 'doc_id': 210137, 'rank': 7, 'score': 0.2958429444537096}, {'query_id': 'Q01', 'doc_id': 211346, 'rank': 8, 'score': 0.28332106845286503}, {'query_id': 'Q01', 'doc_id': 206228, 'rank': 9, 'score': 0.28273256542968167}, {'query_id': 'Q01', 'doc_id': 193353, 'rank': 10, 'score': 0.26677322444052776}, {'query_id': 'Q01', 'doc_id': 198900, 'rank': 11, 'score': 0.26295340313082116}, {'query_id': 'Q01', 'doc_id': 202703, 'rank': 12, 'score': 0.2604626610292017}, {'query_id': 'Q01', 'doc_id': 211970, 'rank

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 7/7 [00:16<00:00,  2.34s/it]


In [20]:
# writing files for trec eval

def file_results(results, filename, run_name):
    with open(filename, "w") as file:
        for r in results:
            file.write(
                f"{r['query_id']} Q0 {r['doc_id']} {r['rank']} {r['score']} {run_name} \n"
            )
            

In [26]:
file_results(results_20, "run_hybrid_k20.txt", "hybrid_k20")
file_results(results_30, "run_hybrid_k30.txt", "hybrid_k30")
file_results(results_50, "run_hybrid_k50.txt", "hybrid_k50")

In [27]:
with open("trec_eval/run_hybrid_k20.txt") as file:
    print(file.read())

Q01 Q0 193378 1 1.0000000119198482 hybrid_k20 
Q01 Q0 193373 2 0.35789587453608174 hybrid_k20 
Q01 Q0 193715 3 0.3210656597166335 hybrid_k20 
Q01 Q0 205685 4 0.306915808957431 hybrid_k20 
Q01 Q0 206230 5 0.3045921282922276 hybrid_k20 
Q01 Q0 193375 6 0.29722598494913616 hybrid_k20 
Q01 Q0 210137 7 0.2958429444537096 hybrid_k20 
Q01 Q0 211346 8 0.28332106845286503 hybrid_k20 
Q01 Q0 206228 9 0.28273256542968167 hybrid_k20 
Q01 Q0 193353 10 0.26677322444052776 hybrid_k20 
Q01 Q0 198900 11 0.26295340313082116 hybrid_k20 
Q01 Q0 202703 12 0.2604626610292017 hybrid_k20 
Q01 Q0 211970 13 0.26031124523978727 hybrid_k20 
Q01 Q0 194660 14 0.2582789037209632 hybrid_k20 
Q01 Q0 205643 15 0.2577703501510312 hybrid_k20 
Q01 Q0 213250 16 0.2565903658959499 hybrid_k20 
Q01 Q0 193402 17 0.25547437045020216 hybrid_k20 
Q01 Q0 210133 18 0.25194235716310376 hybrid_k20 
Q01 Q0 211972 19 0.24938513077244082 hybrid_k20 
Q01 Q0 193386 20 0.24395059234577357 hybrid_k20 
Q02 Q0 213164 1 1.0000000119192933 hybr